In [163]:
import numpy as np 
import pandas as pd
import re
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

[nltk_data] Downloading package punkt_tab to C:\Users\Deep
[nltk_data]     personal\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Deep
[nltk_data]     personal\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# Import the data

In [151]:
credits_ori_df=pd.read_csv("./Data/tmdb_5000_credits.csv")
movies_ori_df=pd.read_csv("./Data/tmdb_5000_movies.csv")
credit_df=credits_ori_df.copy()
movie_df=movies_ori_df.copy()

# Select the required Columns

In [152]:
movie_df=movie_df[["id","keywords","overview"]]

# Merge the columns of both the df's 

In [153]:
merged_df=movie_df.merge(credit_df,left_on="id",right_on="movie_id")

# Data Validation i.e check for null or duplicate

In [154]:
# removing null values
merged_df=merged_df.dropna()
# removing duplicate values
merged_df=merged_df.drop_duplicates()

# Data Processing

### convert the type of keywords, crew and cast columns

In [155]:
# keep only keywords or we can say tags
def keywords_process(text):
    keywords=[]
    arr=json.loads(text)
    for item in arr:
        keywords.append(item["name"].replace(" ",""))
    return keywords

merged_df["keywords"]=merged_df["keywords"].apply(keywords_process)

# keep only top 3 actory names
def cast_process(text):
    final_cast=[]
    arr=json.loads(text)
    count=1
    for item in arr:
        if count <=3:
            name=item["name"]
            name_without_space=name.replace(" ","")
            final_cast.append(name_without_space)
            count+=1
        else:
            break
    return final_cast
merged_df["cast"]=merged_df["cast"].apply(cast_process)

# keep only the director name in the list 
def crew_process(text):
    director_name=[]
    arr=json.loads(text)
    for item in arr:
        if item["job"]=="Director":
            director_name.append(item["name"].replace(" ",""))
            break
    return director_name

merged_df["crew"]=merged_df["crew"].apply(crew_process)

### merge the columns keywords + cast +crew into one 

In [156]:
merged_df["summary"]=merged_df["keywords"]+merged_df["cast"]+merged_df["crew"]
def convert_list_to_str(given_list):
    return " ".join(given_list)

merged_df["summary"]=merged_df["summary"].apply(convert_list_to_str)
merged_df["overview"]=merged_df["overview"]+" "+merged_df["summary"]
merged_df=merged_df[["id","title","overview"]]

# Text Processing on the overview column

In [157]:
# convert all the letters in the lower case 
def convert_lowerCase(text):
    lower_text=str.lower(text)
    return lower_text

# remvoe the !@#$%^&*() from the text
def remove_punctuations(text):
    final_text=re.sub(r"[^A-Za-z0-9\s]","",text)
    return final_text

# remove stop words
def remove_stopwords(text):
    final_str=[]
    stopWords=stopwords.words("english")
    tokens=word_tokenize(text)
    for token in tokens:
        if token in stopWords:
            continue
        else:
            final_str.append(token)
    return " ".join(final_str)        

    
#perfrom stemming on the text
ps=PorterStemmer()
def do_stemming(text):
    stemmed_words=[]
    tokens=word_tokenize(text)
    for token in tokens:
        stemmed_words.append(ps.stem(token))
    return " ".join(stemmed_words)

merged_df["overview"]=merged_df["overview"].apply(convert_lowerCase)
merged_df["overview"]=merged_df["overview"].apply(remove_punctuations)
merged_df["overview"]=merged_df["overview"].apply(remove_stopwords)
merged_df["overview"]=merged_df["overview"].apply(do_stemming)

### perform tokenization on the overview column

In [164]:
vectorization =TfidfVectorizer(max_features=5000)
vectors=vectorization.fit_transform(merged_df["overview"])

### now we have vectors we can use this to find the distance btw them using cosine similarity

array([1.        , 0.        , 0.01798007, ..., 0.04084326, 0.01810331,
       0.        ], shape=(4800,))

In [ ]:
# credit_df.head()
# movie_df.head()
merged_df.head(1)

array([0.01798007, 0.01404035, 1.        , ..., 0.01619313, 0.        ,
       0.        ], shape=(4800,))